In [1]:
import os
import pandas as pd
import anndata as ad

In [3]:
info = pd.read_csv("results/NanoNASCseq_Blastocyst.csv", index_col=0)
cells = info.index.values

for base in ["gene", "transcript"]:
    print("-" * 80)
    print("Level:", base)
    
    prefix = "results/h5ad_raw/Blastocyst"
    path1 = prefix + ".%s_based.filelist.txt" % base
    path2 = prefix + ".%s_based.h5ad" % base
    path3 = prefix + ".%s_based.annotated.h5ad" % base
    path4 = prefix + ".%s_based.counts.csv" % base
    path5 = prefix + ".%s_based.meta.csv" % base

    if not os.path.exists(path1):
        with open(path1, "w+") as fw:
            for cell in cells:
                if base == "gene":
                    path = "../../../1_NanoNASCseq/results/5_expression/4_quant_genes/min_read_2_min_tc_2/%s/%s.tsv" % (cell.split(".")[0], cell)
                else:
                    path = "../../../1_NanoNASCseq/results/8_expression_novel/2_quant_isoforms/min_read_2_min_tc_2/%s/%s.tsv" % (cell.split(".")[0], cell)
                assert os.path.exists(path)
                fw.write(path + "\n")

    if not os.path.exists(path2):
        ! ../../../1_NanoNASCseq/scripts/expression/make_h5ad.py {path1} {path2}

    if not os.path.exists(path3):
        if base == "gene":
            anno = pd.read_csv("../../../1_NanoNASCseq/results/7_assembly_custom/5_gtf_full/MouseBlastocyst.gtf.gene_info.csv", index_col=0)
        else:
            anno = pd.read_csv("../../../1_NanoNASCseq/results/7_assembly_custom/5_gtf_full/MouseBlastocyst.gtf.transcript_info.csv", index_col=0)
        
        adata = ad.read_h5ad(path2)
        print(adata)
        adata.obs = info.loc[adata.obs.index]
        adata = adata[:,adata.var.index.isin(anno.index)]
        adata.var = anno.loc[adata.var.index]
        if base == "gene":
            adata = adata[:,~adata.var["GeneName"].duplicated()].copy()
            adata.var["GeneID"] = adata.var.index
            adata.var.index = adata.var["GeneName"]
            adata.var = adata.var[['GeneID', 'GeneType', 'Chrom', 'Start', 'End', 'Strand']]
        else:
            adata = adata[:,~adata.var["TranscriptName"].duplicated()].copy()
            adata.var["TranscriptID"] = adata.var.index
            adata.var.index = adata.var["TranscriptName"]
            adata.var = adata.var[["TranscriptID", "TranscriptType", "GeneID", "GeneName", "GeneType", "Chrom", "Start", "End", "Strand"]]
        print(adata)
        adata.write(path3, compression="gzip")

    if not os.path.exists(path4):
        mtx = pd.DataFrame(adata.X, index=adata.obs.index, columns=adata.var.index).T
        mtx.to_csv(path4)
        meta = adata.obs
        meta.to_csv(path5)

--------------------------------------------------------------------------------
Level: gene
AnnData object with n_obs × n_vars = 2661 × 27794
    layers: 'new', 'total'
AnnData object with n_obs × n_vars = 2661 × 27524
    obs: 'Run', 'Barcode', 'Species', 'CellType', 's4U', 'Time', 'ActD', 'Phase', 'IAA', 'Stage', 'Cell.Reads', 'Cell.Bases', 'Trimmed.Reads', 'Trimmed.Ratio', 'Mapped.Reads', 'Mapped.Ratio', 'Mito.Ratio', 'Filtered.Reads', 'Filtered.Ratio', 'FilteredClip.Reads', 'FilteredClip.Ratio', 'UMIs', 'mrUMIs', 'Duplicate.Reads', 'Duplicate.Ratio', 'Unique.Reads', 'Genes', 'Isoforms.Assembled', 'Isoforms.Known', 'AC.Ratio', 'AG.Ratio', 'AT.Ratio', 'CA.Ratio', 'CG.Ratio', 'CT.Ratio', 'GA.Ratio', 'GC.Ratio', 'GT.Ratio', 'TA.Ratio', 'TC.Ratio', 'TG.Ratio', 'Pe', 'Pc', 'SNR', 'mrUMIs.New', 'mrUMIs.New.Ratio', 'mrGenes', 'mrGenes.New'
    var: 'GeneID', 'GeneType', 'Chrom', 'Start', 'End', 'Strand'
    layers: 'new', 'total'
-----------------------------------------------------------